# Export fine-tuned Mid-360S PointPillars → Jetson-compatible ONNX

Converts our fine-tuned weights into a TensorRT-ready `pointpillar.onnx` with **our** geometry baked in
(range ±40.96 / z −0.5…3.0, voxel 0.16×0.16×3.5 → 512×512 grid, **256×256** feature map, 3 classes).

### ⚠️ Manual setup BEFORE running (required)
1. **Runtime > Change runtime type**: **Runtime version = 2025.07**, **Hardware accelerator = T4 GPU**
   (ships Python 3.11.13 + torch 2.6.0+cu124, CUDA available). The exporter needs a real GPU
   (`model.cuda()` + GPU NMS during tracing).
2. The trained weights must already be on Drive at
   `/content/drive/MyDrive/perception_ckpts/mid360_portable.pth` (written by the training notebook).
3. Run **top-to-bottom in a FRESH runtime** — the `CUMM_DISABLE_JIT` / `SPCONV_DISABLE_JIT` env vars must be
   set in the first code cell before any spconv import (if spconv was already imported, restart the session).

### Verified environment (same stack as the training run)
Runtime **2025.07** · Python **3.11.13** · torch **2.6.0+cu124** · spconv **2.3.8** (spconv-cu121) ·
numpy **1.23.5** · OpenPCDet **846cf3e** + `patch_openpcdet.py` · CUDA-PointPillars **ce7e2bd** (export tool).


### ✅ Verified working
Runtime **2025.07** · Python **3.11.13** · torch **2.6.0+cu124** · spconv **2.3.8** · numpy **1.23.5** —
export produced `pointpillar.onnx` with outputs **256×256** (`cls_preds` 18, `box_preds` 42,
`dir_cls_preds` 12), geometry verified. Two gotchas baked in below: (1) **restart the kernel** after the
numpy downgrade (Section 3 → resume at Section 4); (2) the ONNX checker must be given the **file path**
(large-model safe), and a harmless `keepdims INT/INTS` warning is expected.

## 1. Environment setup — spconv-cu121 with JIT disabled (the key fix)

Sets `CUMM_DISABLE_JIT` / `SPCONV_DISABLE_JIT` **first**, installs `spconv-cu121` + `numpy<1.24`, and
verifies spconv imports in seconds (no source compile). torch 2.6.0+cu124 ships with the 2025.07 runtime —
we do **not** reinstall it.

In [ ]:
import os, sys, time
# KEY: disable cumm/spconv JIT BEFORE importing them (prevents the failing
# source build — see docs). Must be set first, in a fresh runtime.
os.environ["CUMM_DISABLE_JIT"] = "1"
os.environ["SPCONV_DISABLE_JIT"] = "1"

print("python:", sys.version.split()[0])          # expect 3.11.x
!pip install -q spconv-cu121
!pip install -q "numpy<1.24"

t = time.time()
import spconv
from spconv.utils import Point2VoxelCPU3d
dt = time.time() - t
print(f"spconv {spconv.__version__} imported in {dt:.1f}s | Point2VoxelCPU3d OK")
assert dt < 120, "spconv JIT-compiled — check CUMM_DISABLE_JIT is set FIRST"

import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"

## 2. Clone our repo + mount Drive + stage data

Clones the repo, mounts Drive, asserts the fine-tuned weights exist, and unzips the point-cloud bundle —
`export_onnx.py` needs a `--data_path` directory with a few `.bin` frames to build its `DemoDataset`
(same 48-bin bundle used for training).

In [ ]:
!git clone https://github.com/Abdulrahman2200925/perception.git /content/perception
from google.colab import drive; drive.mount('/content/drive')
import os
WEIGHTS = '/content/drive/MyDrive/perception_ckpts/mid360_portable.pth'
assert os.path.exists(WEIGHTS), f'weights not found at {WEIGHTS} — run the training notebook first.'
print('weights:', WEIGHTS, f'{os.path.getsize(WEIGHTS)/1e6:.1f} MB')

ZIP = '/content/drive/MyDrive/perception_colab_data.zip'
assert os.path.exists(ZIP), f'zip not found at {ZIP} — upload it to Drive root.'
!unzip -q -o "$ZIP" -d /content/perception
import glob
bins = glob.glob('/content/perception/data/derived/bin/**/*.bin', recursive=True)
print('bin files:', len(bins))
assert len(bins) == 48, f'expected 48 bins, found {len(bins)}'
print('data staged OK')

## 3. Clone + PATCH + build OpenPCDet (846cf3e)

`patch_openpcdet.py` fixes THC / `AT_CHECK` / `.data<T>()` for torch 2.x. `requirements.txt` pulls
scikit-image, which drags numpy to 2.x, but 846cf3e uses `np.int`/`np.bool` (removed in 1.24) — so numpy is
force-re-pinned to `1.23.5 --no-deps` **after** requirements.

> ⚠️ **After this cell, do `Runtime > Restart session`, then continue from the guard-rail cell (Section 4).**
> numpy is downgraded on disk but the old 2.0.2 stays loaded in memory until a restart. **Do NOT re-run the
> env/build cells after the restart** — spconv, OpenPCDet, and numpy 1.23.5 are already on disk; the
> guard-rail cell re-sets the JIT env vars, which is enough.

In [ ]:
import os
os.environ["CUMM_DISABLE_JIT"] = "1"; os.environ["SPCONV_DISABLE_JIT"] = "1"  # keep JIT off for any spconv import
!git clone https://github.com/open-mmlab/OpenPCDet.git /content/OpenPCDet
%cd /content/OpenPCDet
!git checkout 846cf3e
!python /content/perception/colab_bundle/patch_openpcdet.py /content/OpenPCDet
!pip install -q -r requirements.txt
!pip install -q "numpy==1.23.5" --force-reinstall --no-deps   # RE-PIN: scikit-image pulls numpy>=1.24 (removed np.int/np.bool); --no-deps stops pip re-upgrading
!python setup.py develop

## 4. Confirm build + versions (guard rail)

**▶️ Resume here after the kernel restart** from Section 3. This cell re-sets the JIT env vars and verifies
pcdet / spconv / numpy 1.23.x.

In [ ]:
import os
os.environ["CUMM_DISABLE_JIT"] = "1"; os.environ["SPCONV_DISABLE_JIT"] = "1"
import pcdet; print("pcdet", pcdet.__version__)
import spconv.pytorch; print("spconv.pytorch OK")
import numpy; print("numpy", numpy.__version__)
assert numpy.__version__.startswith("1.23"), "numpy got bumped — re-pin numpy<1.24 and re-run Cell 3/4"

## 5. Get CUDA-PointPillars (ce7e2bd) + install export tooling

Provides `tool/export_onnx.py` + `tool/modify_onnx.py`. Adds `onnxsim` + `onnx-graphsurgeon` (onnx itself
is already present from the OpenPCDet stack).

In [ ]:
!git clone https://github.com/NVIDIA-AI-IOT/CUDA-PointPillars.git /content/CUDA-PointPillars
%cd /content/CUDA-PointPillars
!git checkout ce7e2bd
!pip install -q onnxsim onnx-graphsurgeon
import onnx, onnxsim, onnx_graphsurgeon
print("onnx", onnx.__version__, "| onnxsim ok | onnx_graphsurgeon ok")

## 6. 🔴 Patch `modify_onnx.py` geometry (KITTI → ours)

The exporter **hardcodes KITTI geometry**: the PPScatter `dense_shape` `[496,432]` and the three output
shapes `248×216`. For our range/voxel the grid is `512×512` and the feature map is `256×256` (channels
18/42/12 are unchanged — same 3 classes × 6 anchors). This patch edits those literals **in place**, from a
pristine `.orig` backup (idempotent), and asserts the edits landed — if the file doesn't match the expected
patterns it prints the relevant lines and **stops** rather than guessing.

In [ ]:
import re, os, shutil, sys

p = '/content/CUDA-PointPillars/tool/modify_onnx.py'
orig = p + '.orig'
if not os.path.exists(orig):
    shutil.copy(p, orig)
src = open(orig).read()   # always patch from pristine .orig -> idempotent

def show(label, text):
    print(label)
    for l in text.splitlines():
        if 'dense_shape' in l or re.search(r'gs\.Variable\(name="(cls_preds|box_preds|dir_cls_preds)"', l):
            print("   ", l.strip())

show("BEFORE:", src)

new = re.sub(r'np\.array\(\[\s*496\s*,\s*432\s*\]\)', 'np.array([512,512])', src)
new = re.sub(r'\(\s*1\s*,\s*248\s*,\s*216\s*,', '(1, 256, 256,', new)

# validate the edits landed; if not, the upstream file changed -> stop, don't guess
ok = ('np.array([512,512])' in new
      and all(f'(1, 256, 256, {ch})' in new for ch in (18, 42, 12))
      and new.count('496') == 0 and new.count('432') == 0
      and new.count('248') == 0 and new.count('216') == 0)
if not ok:
    show("PATTERN MISMATCH — current file:", new)
    raise SystemExit("modify_onnx.py does not match expected KITTI patterns — inspect manually, do not export.")

open(p, 'w').write(new)
show("AFTER:", new)
print("\u2705 modify_onnx.py patched (dense_shape 512x512, outputs 256x256)")

## 7. Wrap the weights for the loader

`export_onnx.py` calls `load_params_from_file`, which reads `checkpoint['model_state']`. Our
`mid360_portable.pth` is a **bare** state_dict (127 keys, no `model_state`), so it is wrapped into
`{'model_state': sd}` and saved in old serialization (portable across torch versions).

In [ ]:
import torch
sd = torch.load('/content/drive/MyDrive/perception_ckpts/mid360_portable.pth',
                map_location='cpu', weights_only=False)
print("bare state_dict keys:", len(sd))
assert not (isinstance(sd, dict) and 'model_state' in sd), "already wrapped? expected a bare state_dict"
torch.save({'model_state': sd}, '/content/mid360_for_export.pth',
           _use_new_zipfile_serialization=False, pickle_protocol=2)
print("wrapped -> /content/mid360_for_export.pth")

## 8. 🔴 Patch `export_onnx.py` for torch 2.6 (positional dict)

torch 2.6 changed `torch.onnx.export` dict handling: a **bare dict** passed as `args` is spread as
**keyword arguments**, so stock `PointPillar.forward(self, batch_dict)` receives no `batch_dict` →
`TypeError: forward() missing 1 required positional argument: 'batch_dict'`. The fix passes the dummy dict
as a **single positional** argument via the documented empty-dict sentinel:
`torch.onnx.export(model, dummy_input, …)` → `torch.onnx.export(model, (dummy_input, {}), …)`.
This changes only *how* `forward` is invoked — **not** the model, the traced ops, the input/output names, or
the 256×256 geometry (which comes from `modify_onnx.py`). Idempotent, with a `.orig` backup.

In [ ]:
import re, os, shutil

p = '/content/CUDA-PointPillars/tool/export_onnx.py'
orig = p + '.orig'
if not os.path.exists(orig):
    shutil.copy(p, orig)
src = open(orig).read()   # always patch from pristine .orig -> idempotent

# locate the export call and its example-input variable (2nd positional arg, on the
# line after `torch.onnx.export(model,`). Detect the var name; do not hardcode.
m = re.search(r'(torch\.onnx\.export\(\s*model\s*,[^\n]*\n\s*)(\w+)(\s*,)', src)
if not m:
    call = re.search(r'torch\.onnx\.export\([^)]*', src)
    print(call.group(0) if call else "no torch.onnx.export call found")
    raise SystemExit("Could not locate the `model, <dummy>,` args pattern — inspect export_onnx.py manually, do not export.")

var = m.group(2)
print("detected example-input var:", var)
print("BEFORE:")
for l in src.splitlines():
    if 'torch.onnx.export' in l or re.match(rf'\s*{var}\s*,', l):
        print("   ", l.rstrip())

new = src[:m.start(2)] + f'({var}, {{}})' + src[m.end(2):]
assert f'({var}, {{}})' in new, "sentinel not inserted — pattern changed"
open(p, 'w').write(new)

print("AFTER:")
for l in new.splitlines():
    if 'torch.onnx.export' in l or f'({var}, {{}})' in l:
        print("   ", l.rstrip())
print(f"\u2705 export_onnx.py patched: {var} -> ({var}, {{}}) (positional dict for torch 2.6)")

## 9. (Fallback — normally SKIP) Return raw head preds if the NMS trace fails

**Only run this if Cell 10 (Run the export) fails *inside* `post_processing` / NMS** (e.g. errors on
NonZero / ScatterND / an iou3d-NMS op while tracing). The positional-dict fix above is usually enough. If
not, this monkeypatches the model's `forward` to run the module list and return the **raw head tensors**
(`cls_preds`, `box_preds`, `dir_cls_preds`) **before** `post_processing` — the same head-output graph
`modify_onnx.py` already cuts to, so the resulting ONNX is unchanged. The cell is **commented out**;
uncomment it and run it **after Cell 7 (build)** and **before Cell 10 (export)**, then re-run the export.

In [ ]:
# FALLBACK — leave commented unless Cell 10 fails inside post_processing/NMS.
# It must be run AFTER the model is built (Cell 8's export uses build_network internally),
# so the cleanest use is: enable this, then re-run the export cell which re-builds + traces.
#
# The export script builds the model itself, so to inject this we patch the detector CLASS
# before export runs. Uncomment the block below, run it, then re-run Cell 10.
#
# import types
# from pcdet.models.detectors.pointpillar import PointPillar
#
# def _export_forward(self, batch_dict):
#     # run VFE -> scatter -> 2D backbone -> dense head; the head writes its raw
#     # preds into self.dense_head.forward_ret_dict during its own forward().
#     for cur_module in self.module_list:
#         batch_dict = cur_module(batch_dict)
#     rd = self.dense_head.forward_ret_dict
#     return rd['cls_preds'], rd['box_preds'], rd['dir_cls_preds']
#
# PointPillar.forward = _export_forward
# print("FALLBACK ENABLED: PointPillar.forward now returns raw head preds (no post_processing).")
#
# Note: with this enabled you can also drop the empty-dict sentinel, but leaving it is harmless.
# Re-run Cell 10 (export) afterwards; Cell 11 asserts still validate the 256x256 geometry.

## 10. Run the export

Run from the repo root so the cfg's **relative** `_BASE_CONFIG_: configs/mid360_dataset.yaml` resolves.
If the trace fails on post-processing / NMS, capture the error verbatim — that is the known open question
(the exporter traces `model.forward` including GPU NMS).

> If this cell fails **inside** `post_processing` / NMS, enable the fallback cell (Section 9) and re-run this cell.


In [ ]:
%cd /content/perception
!mkdir -p /content/export_out
!python /content/CUDA-PointPillars/tool/export_onnx.py \
    --cfg_file configs/mid360_pointpillar.yaml \
    --ckpt /content/mid360_for_export.pth \
    --data_path data/derived/bin/outdoor_20260718_1631 \
    --out_dir /content/export_out
!ls -l /content/export_out

## 11. 🔴 Verify the exported ONNX (do not skip)

Confirms the graph carries **our** geometry — output feature map `256×256` with channels 18/42/12. The
checker is given the **file path** (required for large / >2 GiB models); a harmless `keepdims: INT vs INTS`
warning is an ONNX-opset detail that TensorRT on the Jetson handles — if it ever causes a TRT parse error it
can be fixed later with a one-line graph edit. If a geometry assert fires, the ONNX would silently mis-decode
on the Jetson → **stop and re-check the modify_onnx patch / export cfg.**

In [ ]:
import onnx
path = '/content/export_out/pointpillar.onnx'
# checker must take the PATH for large (>2GiB) models, not the loaded proto
try:
    onnx.checker.check_model(path)
    print("\u2705 onnx.checker passed")
except Exception as e:
    print("\u26a0\ufe0f checker warning (harmless — e.g. keepdims INT/INTS, or >2GiB):", str(e)[:200])
m = onnx.load(path)
print("\ninputs:")
for i in m.graph.input:
    print("  ", i.name, [d.dim_value for d in i.type.tensor_type.shape.dim])
print("outputs:")
for o in m.graph.output:
    print("  ", o.name, [d.dim_value for d in o.type.tensor_type.shape.dim])
shapes = {o.name: [d.dim_value for d in o.type.tensor_type.shape.dim] for o in m.graph.output}
for n, ch in [('cls_preds', 18), ('box_preds', 42), ('dir_cls_preds', 12)]:
    assert n in shapes, f"missing output {n}"
    s = shapes[n]
    assert s[1:] == [256, 256, ch], f"{n} shape {s} != [1,256,256,{ch}]"
print("\n\u2705 ONNX geometry matches our config (256x256)")

## 12. Save the ONNX to Drive

Copies the verified ONNX to Drive with a Mid360-specific name and prints size + md5 — this is the file to
copy to the Jetson.

In [ ]:
import shutil, hashlib, os
src = '/content/export_out/pointpillar.onnx'
dst = '/content/drive/MyDrive/perception_ckpts/pointpillar_mid360.onnx'
shutil.copy(src, dst)
md5 = hashlib.md5(open(dst, 'rb').read()).hexdigest()
print(f"saved -> {dst}")
print(f"size: {os.path.getsize(dst)/1e6:.2f} MB | md5: {md5}")

## 13. Next steps on the Jetson (`iti-es`)

Real paths on the device (`~/Documents/workspaces/perception_ws/src/cuda_pointpillars_ros/`):

1. **Back up the stock ONNX**, then copy ours in:
   ```
   cd ~/Documents/workspaces/perception_ws/src/cuda_pointpillars_ros/model
   cp pointpillar.onnx pointpillar.onnx.kitti.bak
   scp <laptop>:pointpillar_mid360.onnx ./pointpillar.onnx
   ```
2. 🔴 **Delete the stale engine cache** — else the old KITTI engine is deserialized and our ONNX is ignored:
   ```
   rm ~/Documents/workspaces/perception_ws/src/cuda_pointpillars_ros/model/pointpillar.onnx.cache
   ```
3. **Edit `include/params.h`** (current KITTI → our geometry), at these line numbers:
   | line | constant | → required |
   |---|---|---|
   | 25 | `min_x_range` | `-40.96` |
   | 26 | `max_x_range` | `40.96` |
   | 27 | `min_y_range` | `-40.96` |
   | 28 | `max_y_range` | `40.96` |
   | 29 | `min_z_range` | `-0.5` |
   | 30 | `max_z_range` | `3.0` |
   | 34 | `pillar_z_size` | `3.5` |
   | 54 | `anchor_bottom_heights` | `{-0.19,-0.10,-0.10,}` |

   (grid 512² / feature 256² auto-derive; x/y pillar size, classes, anchors[], thresholds unchanged.)
4. **Rebuild** so the new `params.h` compiles into `pc_process`:
   ```
   cd ~/Documents/workspaces/perception_ws
   colcon build --packages-select cuda_pointpillars_ros
   ```
5. **Run the node** (it builds a fresh FP16 engine from the new ONNX) and **replay the staged bag**
   `data/bags/outdoor_20260718_1631`; confirm boxes land on real clusters vs the pre-finetune KITTI baseline.
